In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import html
import ftfy
import string
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 294
Relevant Sources:
Index(['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston',
       'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register',
       'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired',
       'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg',
       'Seattle', 'Ananova', '\N', 'Syfy.com', 'Voice', 'USA', 'Independent',
       'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday',
       'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN',
       'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI',
       'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis',
       'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic',
       'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera',
       'Information', 'IPS', 'TechNewsWorld', 'News24', 'spo

### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
64313                                               Sony Scales Back Plasma in Favor of LCD
53441                                                       Moguls Match Up Over News Corp.
62884                                       Barclays up as Bank of America deal talk mounts
14665                                    Zodiac killer case still fascinates \\n    (AP)\\n
9160                                  Health care is U.S. strategists' top sector: Barron's
23417                              Group Offers Guidelines for Medicare Drug Plan (Reuters)
48222                                              Kennedy cousin appeals murder conviction
28444                                Edwards staffers evacuate office, again \\n    (AP)\\n
8546                                                       Moment of opportunity in Mideast
29834    Spielberg Drops Out as Adviser to Beijing Olympics in Dispute Over D

### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
61541                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               AP - Celebrities were doing last minute primping as they prepared to walk the red carpet at the Golden Globe Awards on Sunday. Jamie Foxx led the contenders with an unprecedented three acting nominations, including best musical or comedy actor for his uncanny portrayal of singer Ray Charles in "Ray."
41876                                                                                              

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number of articles with PageRank 5: 73891
Id
31193    5
8918     5
49106    5
39244    5
11665    5
14527    5
26193    5
50668    5
33708    5
28628    5
Name: page_rank, dtype: int64


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
46166    2007-02-12 15:35:39
77767    2004-11-23 10:09:00
1863     2008-01-03 12:06:20
32049    2007-05-24 20:23:59
65749    2007-02-17 03:55:57
24995    2006-09-15 22:27:54
34388    0000-00-00 00:00:00
45646    2004-11-25 16:56:49
38442    2006-09-03 01:30:51
39406    0000-00-00 00:00:00
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    df['has_date'] = df['dt_obj'].notna().astype(int)
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    df['hour'] = df['dt_obj'].dt.hour.fillna(-1).astype(int)

    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)
new_cols = ['has_date', 'year', 'month', 'day_of_week', 'hour']
print(f"New columns added: {new_cols} , Column removed: ['timestamp']")
print("Test new timestamp features:\n")
print(df[new_cols].sample(10))

New columns added: ['has_date', 'year', 'month', 'day_of_week', 'hour'] , Column removed: ['timestamp']
Test new timestamp features:

       has_date  year  month  day_of_week  hour
Id                                             
53535         0    -1     -1           -1    -1
4195          1  2006     11            1    18
19284         1  2006     12            4     2
18204         1  2007      7            5     6
66909         1  2006     12            6    22
37075         1  2007      2            3    10
21123         0    -1     -1           -1    -1
36531         1  2004     12            0     2
13059         0    -1     -1           -1    -1
30220         1  2007     11            3    20


### *Title* feature stemming

In [9]:
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = text.lower() 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    # tag Money
    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    # tag Percentage
    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    # tag Score
    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    #tag Date
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random titles\n")
    
    for idx, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text:  {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_simple(df['title'], clean_title, n=10)

Test on 10 random titles

Original text:  138 ultras surrender in Tripura
Processed text: ultra surrend tripura
--------------------------------------------------
Original text:  Agassi committed to another full year on circuit
Processed text: agassi commit anoth full year circuit
--------------------------------------------------
Original text:  Airstrike said to kill 50 at suspected Qaeda site
Processed text: airstrik said kill suspect qaeda site
--------------------------------------------------
Original text:  Q&A: Cisco VP Talks RFID
Processed text: cisco talk rfid
--------------------------------------------------
Original text:  Taco Bell parent says E.coli no longer in outlets
Processed text: taco bell parent say coli longer outlet
--------------------------------------------------
Original text:  Israel Bombs Hamas Buildings in Gaza, Killing 5
Processed text: israel bomb hama build gaza kill
--------------------------------------------------
Original text:  When ETFs Are Bette

### *Article* feature stemming

In [10]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = re.sub(r'<[^>]+>', ' ', text)
    text = ftfy.fix_text(text)
    text = text.strip()
    
    text = re.sub(r'^\s*[A-Z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)
    text = re.sub(r'^\s*[A-Z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)

    text = text.lower()
    text = text[:500]

    # tag Money
    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    # tag Percentage
    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    # tag Score
    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    #tag Date
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random articles\n")
    for idx, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_simple_article(df['article'], clean_article, n=10)

Test on 10 random articles

Original text (First 200 char): KABUL, Afghanistan - An independent commission will probe claims by all 15 challengers to interim leader Hamid Karzai that Afghanistan's first direct presidential election was marred by incompetence and fraud, a top official said Sunday.    The move to head off the attack on the vote's legitimacy came as workers began the long process of collecting ballots from Saturday's historic election, in which Karzai was a heavy favorite...
Processed text: independ commiss probe claim challeng interim leader hamid karzai afghanistan first direct presidenti elect mar incompet fraud top offici said sunday move head attack vote legitimaci came worker began long process collect ballot saturday histor elect karzai heavi favorit
--------------------------------------------------
Original text (First 200 char): http://www.StrutYourHut.com  launches as a fun Real Estate site where the creators say, "Real Estate  Gets Personal." StrutYourHut.comS

### *Title + Article* features merge

In [11]:
print(f"Starting shape: {df.shape}")
print(f"Starting columns: {df.columns.tolist()}")
df['title_clean'] = df['title'].apply(clean_title)
df['article_clean'] = df['article'].apply(clean_article)
df['text_combined'] = (df['title_clean'].fillna('') + " " + df['article_clean'].fillna('')).str.strip()

n_empty = (df['text_combined'] == "").sum()
print(f"Removing {n_empty} rows with empty text")
df = df[df['text_combined'] != ""]
df = df.drop(columns=['title', 'article', 'title_clean', 'article_clean'])

print(f"Final shape: {df.shape}")
print(f"Actual columns: {df.columns.tolist()}")
print("Example of title + article combined:")
print(df['text_combined'].iloc[0])

Starting shape: (79997, 10)
Starting columns: ['source', 'title', 'article', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour']
Removing 3 rows with empty text
Final shape: (79994, 9)
Actual columns: ['source', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour', 'text_combined']
Example of title + article combined:
opec boost nigeria oil revenu tag_money bpd organis petroleum export countri opec hike offici output one million barrel per day effect novemb nigeria get barrel per day per cent new quota
